# QLoRA eval-only from checkpoint

This notebook restores a Qwen model with a LoRA adapter from checkpoint files and computes validation/test metrics.

Expected `instruction_temporal` files:

```text
train.jsonl
val.jsonl
test.jsonl
```

Expected checkpoint files:

```text
adapter_model.safetensors
adapter_config.json
optimizer.pt
scheduler.pt
scaler.pt
rng_state.pth
trainer_state.json
training_args.bin
```

Only `adapter_model.safetensors` and `adapter_config.json` are required for evaluation.


In [ ]:
!pip install -q -U "transformers>=4.45.0" accelerate peft bitsandbytes scikit-learn pandas pyarrow safetensors


In [ ]:
import os, json, glob, math, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("device count:", torch.cuda.device_count())


## 1. Конфиг

По умолчанию evaluation-size такой же, как был в большом pilot-запуске:

```text
EVAL_PER_CLASS = 1000
```

То есть 2000 val и 2000 test.


In [ ]:
MODEL_NAME_FALLBACK = "Qwen/Qwen3-4B-Instruct-2507"
EVAL_PER_CLASS = 1000
MAX_SEQ_LENGTH = 1024

# Если val уже известен и надо сэкономить время, можно поставить COMPUTE_VAL = False
# и задать MANUAL_THRESHOLD = 0.35 из предыдущего вывода.
COMPUTE_VAL = True
MANUAL_THRESHOLD = 0.35

RUN_NAME = f"qwen_checkpoint_eval_eval{EVAL_PER_CLASS*2}"
OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)


## 2. Поиск файлов в Kaggle input


In [ ]:
def find_file(filename, prefer_substr=None):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"Не найден {filename} в /kaggle/input")
    if prefer_substr:
        hits = sorted(hits, key=lambda p: (prefer_substr.lower() not in p.lower(), len(p)))
    else:
        hits = sorted(hits, key=lambda p: len(p))
    return Path(hits[0])

train_path = find_file("train.jsonl", "instruction")
val_path = find_file("val.jsonl", "instruction")
test_path = find_file("test.jsonl", "instruction")
adapter_config_path = find_file("adapter_config.json")
adapter_dir = adapter_config_path.parent

print("train:", train_path)
print("val:", val_path)
print("test:", test_path)
print("adapter_dir:", adapter_dir)
print("adapter files:", [p.name for p in adapter_dir.iterdir() if p.is_file()][:20])


In [ ]:
with open(adapter_config_path, "r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)

print(json.dumps({k: adapter_cfg.get(k) for k in ["base_model_name_or_path", "peft_type", "task_type", "r", "lora_alpha", "target_modules"]}, ensure_ascii=False, indent=2))

MODEL_NAME = adapter_cfg.get("base_model_name_or_path") or MODEL_NAME_FALLBACK
if MODEL_NAME.startswith("/") or MODEL_NAME.startswith("./"):
    MODEL_NAME = MODEL_NAME_FALLBACK
print("MODEL_NAME:", MODEL_NAME)


## 3. Загрузка данных и balanced samples


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

val_df = load_jsonl(val_path)
test_df = load_jsonl(test_path)

print("val/test shapes:", val_df.shape, test_df.shape)
print("val output:", val_df["output"].value_counts(dropna=False).to_dict())
print("test output:", test_df["output"].value_counts(dropna=False).to_dict())

def make_balanced_sample(df, n_per_class, seed=42):
    out_col = df["output"].astype(str).str.strip().str.lower()
    yes = df[out_col == "yes"].sample(n=n_per_class, random_state=seed)
    no = df[out_col == "no"].sample(n=n_per_class, random_state=seed)
    return pd.concat([yes, no], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

val_pilot_df = make_balanced_sample(val_df, EVAL_PER_CLASS, SEED)
test_pilot_df = make_balanced_sample(test_df, EVAL_PER_CLASS, SEED)

print("val_pilot:", val_pilot_df.shape, val_pilot_df["output"].value_counts().to_dict())
print("test_pilot:", test_pilot_df.shape, test_pilot_df["output"].value_counts().to_dict())


## 4. Tokenizer и prompt builder


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = "You are a recommendation model. Answer only Yes or No."

def normalize_answer(x):
    x = str(x).strip()
    if x.lower().startswith("yes"):
        return "Yes"
    if x.lower().startswith("no"):
        return "No"
    raise ValueError(f"Bad output: {x}")

def build_user_text(row):
    instruction = str(row.get("instruction", "")).strip()
    inp = str(row.get("input", "")).strip()
    if instruction and inp:
        return instruction + "\n\n" + inp
    if "prompt_text" in row and pd.notna(row["prompt_text"]):
        return str(row["prompt_text"]).strip()
    raise ValueError("Нет instruction/input и нет prompt_text")

def make_prompt_text(row):
    user_text = build_user_text(row)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(make_prompt_text(val_pilot_df.iloc[0])[-1500:])
print("true:", normalize_answer(val_pilot_df.iloc[0]["output"]))


## 5. Проверка длины prompt-ов


In [ ]:
def prompt_len(row):
    ids = tokenizer(make_prompt_text(row), add_special_tokens=False).input_ids
    return len(ids)

lens = [prompt_len(val_pilot_df.iloc[i]) for i in range(min(500, len(val_pilot_df)))]
print("val length min/mean/max:", min(lens), float(np.mean(lens)), max(lens))
print("would truncate >", MAX_SEQ_LENGTH - 4, ":", sum(x > MAX_SEQ_LENGTH - 4 for x in lens), "of", len(lens))


## 6. Загрузка base model + adapter


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.config.use_cache = False

model = PeftModel.from_pretrained(base_model, str(adapter_dir))
model.eval()
print("Loaded adapter from:", adapter_dir)


## 7. Evaluation через logprob Yes/No


In [ ]:
@torch.no_grad()
def answer_logprob(model, prompt_text, answer_text):
    model.eval()
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    answer_ids = tokenizer(answer_text, add_special_tokens=False).input_ids
    max_prompt_len = MAX_SEQ_LENGTH - len(answer_ids)
    if len(prompt_ids) > max_prompt_len:
        prompt_ids = prompt_ids[-max_prompt_len:]
    input_ids = torch.tensor([prompt_ids + answer_ids], dtype=torch.long, device=model.device)
    attention_mask = torch.ones_like(input_ids)
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[0]
    start = len(prompt_ids)
    total = 0.0
    for j, tok_id in enumerate(answer_ids):
        pos = start + j - 1
        log_probs = torch.log_softmax(logits[pos], dim=-1)
        total += float(log_probs[tok_id].detach().cpu())
    return total

@torch.no_grad()
def predict_scores(model, df):
    scores, y_true, sample_ids = [], [], []
    yes_answer = "Yes" + tokenizer.eos_token
    no_answer = "No" + tokenizer.eos_token
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        prompt_text = make_prompt_text(row)
        lp_yes = answer_logprob(model, prompt_text, yes_answer)
        lp_no = answer_logprob(model, prompt_text, no_answer)
        m = max(lp_yes, lp_no)
        p_yes = math.exp(lp_yes - m) / (math.exp(lp_yes - m) + math.exp(lp_no - m))
        scores.append(p_yes)
        y_true.append(1 if normalize_answer(row["output"]) == "Yes" else 0)
        sample_ids.append(row.get("sample_id", None))
        if i % 250 == 0:
            print("processed", i, "/", len(df))
    return np.array(y_true), np.array(scores), sample_ids

def best_threshold_by_f1(y_true, scores):
    best = {"threshold": 0.5, "f1": -1}
    for thr in np.linspace(0.01, 0.99, 99):
        pred = (scores >= thr).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(thr), "precision": float(p), "recall": float(r), "f1": float(f1)}
    return best

def calc_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "threshold": float(threshold)
    }

def evaluate_split(model, df, name, threshold=None):
    y_true, scores, sample_ids = predict_scores(model, df)
    if threshold is None:
        threshold = best_threshold_by_f1(y_true, scores)["threshold"]
    metrics = calc_metrics(y_true, scores, threshold)
    print(name, metrics)
    pred_df = pd.DataFrame({
        "sample_id": sample_ids,
        "y_true": y_true,
        "score_yes": scores,
        "pred": (scores >= threshold).astype(int)
    })
    return metrics, pred_df


## 8. Считаем val и test


In [ ]:
if COMPUTE_VAL:
    qlora_val_metrics, qlora_val_pred = evaluate_split(model, val_pilot_df, "qlora_val_pilot")
    thr = qlora_val_metrics["threshold"]
else:
    qlora_val_metrics, qlora_val_pred = None, None
    thr = MANUAL_THRESHOLD
    print("Using manual threshold:", thr)

qlora_test_metrics, qlora_test_pred = evaluate_split(model, test_pilot_df, "qlora_test_pilot", threshold=thr)

rows = []
if qlora_val_metrics is not None:
    rows.append({"stage": "qlora_checkpoint", "split": "val_pilot", **qlora_val_metrics})
rows.append({"stage": "qlora_checkpoint", "split": "test_pilot", **qlora_test_metrics})
summary = pd.DataFrame(rows)
summary


## 9. Сохранение результатов


In [ ]:
summary.to_csv(OUT_DIR / "qwen_checkpoint_eval_metrics.csv", index=False)
if qlora_val_pred is not None:
    qlora_val_pred.to_csv(OUT_DIR / "qwen_checkpoint_val_predictions.csv", index=False)
qlora_test_pred.to_csv(OUT_DIR / "qwen_checkpoint_test_predictions.csv", index=False)

run_summary = {
    "model_name": MODEL_NAME,
    "adapter_dir": str(adapter_dir),
    "eval_per_class": int(EVAL_PER_CLASS),
    "max_seq_length": int(MAX_SEQ_LENGTH),
    "compute_val": bool(COMPUTE_VAL),
    "manual_threshold": float(MANUAL_THRESHOLD),
    "metrics": summary.to_dict(orient="records")
}
with open(OUT_DIR / "qwen_checkpoint_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

shutil.make_archive(str(OUT_DIR / "qwen_checkpoint_eval_outputs"), "zip", OUT_DIR)
print("saved:", OUT_DIR)
print("archive:", OUT_DIR / "qwen_checkpoint_eval_outputs.zip")


## 10. Output artifacts

Evaluation artifacts:

```text
qwen_checkpoint_eval_outputs.zip
qwen_checkpoint_eval_metrics.csv
qwen_checkpoint_test_predictions.csv
qwen_checkpoint_eval_summary.json
```

Optional validation predictions:

```text
qwen_checkpoint_val_predictions.csv
```
